<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드, 저자: <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 미세조정된 모델 로딩 및 사용

이 노트북은 7장에서 [ch07.ipynb](ch07.ipynb)를 통해 지시사항 미세조정되고 저장된 미세조정 모델을 로딩하는 최소한의 코드를 포함합니다.

In [ ]:
from importlib.metadata import version

pkgs = [
    "tiktoken",    # 토크나이저(Tokenizer)
    "torch",       # 딥러닝 라이브러리(Deep learning library)
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
from pathlib import Path

finetuned_model_path = Path("gpt2-medium355M-sft.pth")
if not finetuned_model_path.exists():
    print(
        f"'{finetuned_model_path}'를 찾을 수 없습니다.\n"
        "미세조정된 모델을 미세조정하고 저장하려면 `ch07.ipynb` 노트북을 실행하세요."
    )

In [ ]:
from previous_chapters import GPTModel


BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘 크기(Vocabulary size)
    "context_length": 1024,  # 컨텍스트 길이(Context length)
    "drop_rate": 0.0,        # 드롭아웃 비율(Dropout rate)
    "qkv_bias": True         # Query-key-value 편향(Query-key-value bias)
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
model = GPTModel(BASE_CONFIG)

In [ ]:
import torch

model.load_state_dict(torch.load(
    "gpt2-medium355M-sft.pth",
    map_location=torch.device("cpu"),
    weights_only=True
))
model.eval();

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
prompt = """다음은 작업을 설명하는 지시사항입니다. 요청을 적절히 완성하는 응답을 작성하세요.

### Instruction:
능동태 문장을 수동태로 변환하세요: 'The chef cooks the meal every day.'
"""

In [ ]:
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)

def extract_response(response_text, input_text):
    return response_text[len(input_text):].replace("### Response:", "").strip()

torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(prompt, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

response = token_ids_to_text(token_ids, tokenizer)
response = extract_response(response, prompt)
print(response)